Data Preprocessing

In [ ]:
import glob
import os


FREE MASK

In [ ]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities.
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyyaml: filename=PyYAML-5.1-cp310-cp310-linux_x86_64.whl size=44090 sha256=13f20aa359bdfb41a977c1d4c32c80d9f3748e3845656901a9b983ba171bdae1
  Stored in directory: /root/.cache/pip/wheels/70/83/31/975b737609aba39a4099d471d5684141c1fdc3404f97e7f68a
Successfully built pyyaml
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0
    Uninstalling PyYAML-6.0:
      Successfully uninstalled PyYAML-6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask 2022.12.1 requires pyyaml>=5.3.1, but you have pyyaml 5.1 which is incompatible.
flax 0.6.9 requires PyYAML>=5.4.1, but you have pyyaml 5.1 which is incompatibl

No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 12.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61405 sha256=35cb08e97b3f1dd645a510140e90753587b93c4f83be6fae30d67372879e9d93
  Stored in directory: /root/.cache/pip/wheels/01/c0/af/77c1cf53a1be9e42a52b48e5af2169d40ec2e89f7362489dd0
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144554 sha256=03ee856c0dd3533a68e1058b5b3d2b

In [ ]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0
torch:  2.0 ; cuda:  cu118
detectron2: 0.6


In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [ ]:
!unzip free_solo.zip

Archive:  free_solo.zip
   creating: FreeSOLO/
   creating: FreeSOLO/demo/
  inflating: FreeSOLO/demo/predictor.py  
  inflating: FreeSOLO/demo/merge_json.py  
  inflating: FreeSOLO/demo/inference_freemask.py  
  inflating: FreeSOLO/demo/pipeline.jpg  
  inflating: FreeSOLO/demo/vis.jpg   
  inflating: FreeSOLO/demo/demo.py   
  inflating: FreeSOLO/inference_freemask.sh  
   creating: FreeSOLO/tools/
  inflating: FreeSOLO/tools/gen_pseudo_labels.py  
  inflating: FreeSOLO/tools/visualize_data.py  
  inflating: FreeSOLO/tools/eval_cocoapi.py  
  inflating: FreeSOLO/tools/convert-pretrain-to-detectron2.py  
  inflating: FreeSOLO/tools/split_coco_train_sup10_usemask.py  
  inflating: FreeSOLO/tools/visualize_json_results.py  
  inflating: FreeSOLO/tools/split_coco_train_sup10.py  
  inflating: FreeSOLO/tools/merge_train2017_unlabeled2017.py  
  inflating: FreeSOLO/tools/visualize.sh  
  inflating: FreeSOLO/train.sh       
  inflating: FreeSOLO/LICENSE        
   creating: FreeSOLO/freesol

In [ ]:
!ls

detectron2  FreeSOLO  free_solo.zip  sample_data


In [ ]:
# Copyright (c) 2021-2022, NVIDIA Corporation & Affiliates. All rights reserved.
#
# This work is made available under the Nvidia Source Code License-NC.
# To view a copy of this license, visit
# https://github.com/NVlabs/FreeSOLO/blob/main/LICENSE

import argparse
import glob
import multiprocessing as mp
import time
import tqdm
import json
import pycocotools.mask as mask_util

import numpy as np
import torch
import torch.nn.functional as F

from detectron2.config import get_cfg
from detectron2.data.detection_utils import read_image
from detectron2.utils.logger import setup_logger

# from FreeSOLO/demo/predictor.py import VisualizationDemo

from FreeSOLO.demo.predictor import VisualizationDemo

import sys
sys.path.append('.')

from FreeSOLO.freesolo import add_solo_config

from FreeSOLO.freesolo.modeling.solov2.utils import matrix_nms, center_of_mass

# constants
WINDOW_NAME = "COCO detections"


def setup_cfg(args):
    # load config from file and command-line arguments
    cfg = get_cfg()
    add_solo_config(cfg)
    # To use demo for Panoptic-DeepLab, please uncomment the following two lines.
    # from detectron2.projects.panoptic_deeplab import add_panoptic_deeplab_config  # noqa
    # add_panoptic_deeplab_config(cfg)
    cfg.merge_from_file(args.config_file)
    cfg.merge_from_list(args.opts)
    # Set score_threshold for builtin models
    cfg.freeze()
    return cfg


def get_parser():
    parser = argparse.ArgumentParser(description="Detectron2 demo for builtin configs")
    parser.add_argument(
        "--config-file",
        default="configs/quick_schedules/mask_rcnn_R_50_FPN_inference_acc_test.yaml",
        metavar="FILE",
        help="path to config file",
    )
    parser.add_argument("--webcam", action="store_true", help="Take inputs from webcam.")
    parser.add_argument("--video-input", help="Path to video file.")
    parser.add_argument(
        "--input",
        #nargs="+",
        help="A list of space separated input images; "
        "or a single glob pattern such as 'directory/*.jpg'",
    )
    parser.add_argument(
        "--output",
        help="A file or directory to save output visualizations. "
        "If not given, will show output in an OpenCV window.",
    )

    parser.add_argument(
        "--split",
        type=int,
        default=0,
        help="Split id.",
    )
    parser.add_argument(
        "--opts",
        help="Modify config options using the command-line 'KEY VALUE' pairs",
        default=[],
        nargs=argparse.REMAINDER,
    )
    return parser


if __name__ == "__main__":
    print("MIAN")
    mp.set_start_method("spawn", force=True)
    print("mp start")
    args = get_parser().parse_args()
    print(args)
    setup_logger(name="fvcore")
    logger = setup_logger()
    logger.info("Arguments: " + str(args))

    cfg = setup_cfg(args)

    demo = VisualizationDemo(cfg)


    ann_dict = dict()
    ann_dict_ = json.load(open('datasets/coco/annotations/instances_train2017.json'))
    ann_dict['categories'] = ann_dict_['categories']

    images_list = []
    anns_list = []
    ann_id = 0

    paths = glob.glob(args.input + '/*g')
    split = args.split
    if split == -1:
        save_path = args.output
        cur_paths = paths
    else:
        num_each_split = 15000
        image_range = [split*num_each_split, (split+1) * num_each_split]
        save_path = args.output + str(split)
        assert image_range[0] < len(paths)
        cur_paths = paths[image_range[0]:min(len(paths), image_range[1])]
    for path in tqdm.tqdm(cur_paths, disable=not args.output):
        # use PIL, to be consistent with evaluation
        try:
            img = read_image(path, format="BGR")
        except:
            continue
        height, width, _ = img.shape
        img_path = path
        start_time = time.time()
        predictions, visualized_output = demo.run_on_image(img)

        keys = predictions['res5'][0]
        scale_factors = [1.0, 0.5, 0.25]
        queries_list = []
        for scale_factor in scale_factors:
            cur_queries = F.interpolate(keys[None, ...], scale_factor=scale_factor, mode='bilinear')[0].reshape(keys.shape[0], -1).permute(1, 0)
            num_q = len(cur_queries)
            queries_list.append(cur_queries)
        queries = torch.cat(queries_list)
        _, H, W = keys.shape
        keys = keys / keys.norm(dim=0, keepdim=True)
        queries = queries / queries.norm(dim=1, keepdim=True)
        attn = queries @ keys.reshape(keys.shape[0], -1)
        # normalize
        attn -= attn.min(-1, keepdim=True)[0]
        attn /= attn.max(-1, keepdim=True)[0]

        attn = attn.reshape(attn.shape[0], H, W)

        soft_masks = attn
        masks = soft_masks >= 0.5

        # downsample queries
        queries = F.interpolate(queries[None, ...], size=128, mode='linear')[0]

        sum_masks = masks.sum((1,2))
        keep = sum_masks > 1
        if keep.sum() == 0:
            continue
        masks = masks[keep]
        soft_masks = soft_masks[keep]
        sum_masks = sum_masks[keep]
        queries = queries[keep]

        # Matrix NMS
        maskness = (soft_masks * masks.float()).sum((1, 2)) / sum_masks
        sort_inds = torch.argsort(maskness, descending=True)
        maskness = maskness[sort_inds]
        masks = masks[sort_inds]
        sum_masks = sum_masks[sort_inds]
        soft_masks = soft_masks[sort_inds]
        queries = queries[sort_inds]
        maskness = matrix_nms(maskness*0, masks, sum_masks, maskness, sigma=2, kernel='gaussian')

        sort_inds = torch.argsort(maskness, descending=True)
        if len(sort_inds) > 20:
            sort_inds = sort_inds[:20]
        masks = masks[sort_inds]
        maskness = maskness[sort_inds]
        soft_masks = soft_masks[sort_inds]
        queries = queries[sort_inds]

        soft_masks = F.interpolate(soft_masks[None, ...], size=(height, width), mode='bilinear')[0]
        masks = (soft_masks >= 0.5).float()
        sum_masks = masks.sum((1, 2))

        # mask to box
        width_proj = masks.max(1)[0]
        height_proj = masks.max(2)[0]
        box_width, box_height = width_proj.sum(1), height_proj.sum(1)
        center_ws, _ = center_of_mass(width_proj[:, None, :])
        _, center_hs = center_of_mass(height_proj[:, :, None])
        boxes = torch.stack([center_ws-0.5*box_width, center_hs-0.5*box_height, center_ws+0.5*box_width, center_hs+0.5*box_height], 1)
        #boxes = []
        #for mask in masks.cpu().numpy():
        #    ys, xs = np.where(mask)
        #    box = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]
        #    boxes.append(box)
        #boxes = torch.tensor(boxes, device = maskness.device)

        # filter masks on the top border or with large width
        keep = center_hs > 0.2 * height
        keep_2 = (boxes[:, 2] - boxes[:, 0]) < 0.95 * width
        keep_3 = maskness >= 0.7
        keep = keep & keep_2 & keep_3
        #
        if keep.sum() == 0:
            continue
        masks = masks[keep]
        maskness = maskness[keep]
        boxes = boxes[keep]
        queries = queries[keep]

        # coco format
        img_name = img_path.split('/')[-1].split('.')[0]
        try:
            img_id = int(img_name)
        except:
            img_id = int(img_name.split('_')[-1])
        cur_image_dict = {'file_name': img_path.split('/')[-1],
                          'height': height,
                          'width': width,
                          'id':  img_id}
        images_list.append(cur_image_dict)


        masks = masks.cpu().numpy()
        maskness = maskness.cpu().numpy()
        boxes = boxes.tolist()
        queries = queries.tolist()
        rles = [mask_util.encode(np.array(mask[:, :, None], order="F", dtype="uint8"))[0]
                            for mask in masks]
        for idx in range(len(masks)):
            rle = rles[idx]
            rle['counts'] = rle['counts'].decode('ascii')
            cur_ann_dict = {'segmentation': rle,
                            'bbox': boxes[idx],
                            'score': float(maskness[idx]),
                            'emb': queries[idx],
                            'iscrowd': 0,
                            'image_id': img_id,
                            'category_id': 1,
                            'id':  ann_id}
            ann_id += 1
            anns_list.append(cur_ann_dict)


    ann_dict['images'] = images_list
    ann_dict['annotations'] = anns_list
    json.dump(ann_dict, open(save_path, 'w'))
    #json.dump(anns_list, open(save_path+'ann', 'w'))
    print("Done: {} images, {} annotations.".format(len(images_list), len(anns_list)))

MIAN
mp start


usage: ipykernel_launcher.py [-h] [--config-file FILE] [--webcam]
                             [--video-input VIDEO_INPUT] [--input INPUT]
                             [--output OUTPUT] [--split SPLIT] [--opts ...]
ipykernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-89935750-032f-4c8f-b5ac-0fafafa1593b.json


SystemExit: ignored

/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
